In [ ]:
# =============================================================================
# 模块 1: 导入模块、设定超参数与文件路径 (ChemBERTa版)
# =============================================================================

import os
import json
import pandas as pd
from rdkit import Chem
from IPython.display import display, HTML, clear_output
import numpy as np
from tqdm.notebook import tqdm
import torch 
from torch_geometric.data import Data
import utils
import io
from contextlib import redirect_stderr
import re 
from openbabel import pybel  
from transformers import AutoTokenizer, AutoModel
import itertools

# --- 固定随机种子 ---
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"全局随机种子已固定为: {RANDOM_SEED}")

# --- Embedding 维度设置 ---
RAW_EMBEDDING_DIM = 768 
TARGET_EMBEDDING_DIM = 24  
EMBEDDING_DIM = TARGET_EMBEDDING_DIM 

# --- 超参数设定 ---
NUM_MOLECULES_TO_PROCESS = None

# 1. 输入路径: 
input_file_path = os.path.join('datas', 'tain_datas', 'GATv2', 'smiles', 'smiles.xlsx')

# 2. 输出基础目录
output_dir_base = os.path.join('processed_data')

# PyG 数据输出目录: GNN/CHEMCUT/processed_data/GATv2_datas
PYG_DATA_DIR = os.path.join(output_dir_base, 'GATv2_datas') 
os.makedirs(PYG_DATA_DIR, exist_ok=True) 

# 3. 词汇表与统计量 (指向 GNN/CHEMCUT/processed_data 下的文件)
VOCAB_FILE = os.path.join(output_dir_base, 'fragment_vocabulary.json')
NORM_STATS_FILE = os.path.join(output_dir_base, 'normalization_stats.json')

print(f"输入文件: {input_file_path}")
print(f"PyG数据输出目录: {PYG_DATA_DIR}")
print(f"词汇表与统计量读取目录: {output_dir_base}")

# =============================================================================
# [新增] ChemBERTa 模型初始化与降维矩阵
# =============================================================================

HF_MODEL_NAME = "seyonec/ChemBERTa-Zinc-Base-v1"
LOCAL_MODEL_PATH = os.path.join(os.getcwd(), "ChemBERTa_Local")

# 1. 检查本地模型
if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"[ChemBERTa] 本地未检测到模型，请先确保已下载模型至: {LOCAL_MODEL_PATH}")
else:
    print(f"[ChemBERTa] 使用本地模型: {LOCAL_MODEL_PATH}")

# 2. 加载模型
print(f"[ChemBERTa] 正在加载模型权重...")
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
model = AutoModel.from_pretrained(LOCAL_MODEL_PATH)
model.eval()
print(f"[ChemBERTa] 加载完成。")

# 3. 初始化固定随机投影矩阵
if TARGET_EMBEDDING_DIM < RAW_EMBEDDING_DIM:
    print(f"[Embedding] 初始化随机投影矩阵: {RAW_EMBEDDING_DIM} -> {TARGET_EMBEDDING_DIM}")
    # [关键] 强制重置种子以保证矩阵一致性
    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    _proj = torch.randn(RAW_EMBEDDING_DIM, TARGET_EMBEDDING_DIM)
    PROJECTION_MATRIX = _proj / torch.norm(_proj, dim=0)
else:
    print(f"[Embedding] 目标维度 >= 原生维度，不进行压缩。")
    PROJECTION_MATRIX = None

# --- 状态加载 (词汇表与统计量) ---

# 1. 加载词汇表
try:
    with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)
    vocabulary = set(vocab_data.keys())
    print(f"成功加载词汇表，大小: {len(vocabulary)}")
except (FileNotFoundError, json.JSONDecodeError):
    vocab_data = {}
    vocabulary = set()
    print(f"未找到或词汇表文件为空，将创建一个新的。")

# 2. 加载归一化统计信息 
try:
    with open(NORM_STATS_FILE, 'r', encoding='utf-8') as f:
        norm_stats = json.load(f)
    print(f"成功加载归一化参数。")
except (FileNotFoundError, json.JSONDecodeError):
    norm_stats = {}
    print(f"警告: 归一化参数文件为空或已损坏。")

In [ ]:
# =============================================================================
# 模块 2: 加载和准备分子数据
# =============================================================================

try:
    df_full = pd.read_excel(input_file_path)
    activity_columns = [col for col in df_full.columns if col.startswith('分子活性')]
    required_columns = ['分子ID', '分子smiles式'] + activity_columns
    if not all(col in df_full.columns for col in ['分子ID', '分子smiles式']):
        raise ValueError("输入文件中缺少 '分子ID' 或 '分子smiles式' 列。")
    df = df_full[required_columns].dropna(subset=['分子ID', '分子smiles式']).copy()
    df_to_process = df.head(NUM_MOLECULES_TO_PROCESS)
    molecules_data = df_to_process.to_dict('records')
    print(f"成功加载数据。共发现 {len(df)} 个分子，本次将处理 {len(molecules_data)} 个。")
    print(f"识别到的活性数据列: {activity_columns}")
except FileNotFoundError:
    print(f"错误: 未在指定路径找到输入文件 '{input_file_path}'。")
    molecules_data = []
except Exception as e:
    print(f"读取或处理数据时发生错误: {e}")
    molecules_data = []

In [ ]:
# =============================================================================
# 模块 3 (同步修订版): 循环处理每个分子 (适配 utils.py 新逻辑)
# =============================================================================

all_fragments_this_run = []
all_molecules_processed_data = []

# 统计计数器
stats_counter = {
    'total': 0,
    'success': 0,
    'failed_load': 0,
    'failed_preprocess': 0,
    'failed_quality_control': 0  # [新增] 因丢失片段过多而丢弃
}

print("开始处理分子...")

for mol_data in tqdm(molecules_data, desc="Processing Molecules"):
    stats_counter['total'] += 1
    compound_id = mol_data['分子ID'] 
    smiles_string = mol_data['分子smiles式']
    
    try:
        # --- 1. 分子加载与 OpenBabel 修复 (保持原逻辑) ---
        rdkit_error_stream = io.StringIO()
        mol = None 
        
        with redirect_stderr(rdkit_error_stream):
            if smiles_string and isinstance(smiles_string, str):
                mol = Chem.MolFromSmiles(smiles_string)
        
        if not mol and smiles_string and isinstance(smiles_string, str):
            try:
                ob_mol = pybel.readstring("smi", smiles_string)
                fixed_smi = ob_mol.write("can").strip()
                with redirect_stderr(rdkit_error_stream):
                    mol_fixed = Chem.MolFromSmiles(fixed_smi)
                if mol_fixed:
                    mol = mol_fixed
                    smiles_string = fixed_smi 
            except Exception:
                pass

        if not mol:
            stats_counter['failed_load'] += 1
            continue

        # [新增修订] 强制 Canonical 化并更新数据源
        try:
            canonical_smi = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
            mol_data['分子smiles式'] = canonical_smi
            smiles_string = canonical_smi 
            mol = Chem.MolFromSmiles(canonical_smi)
            if mol is None: raise ValueError("Reload failed")
        except Exception:
            stats_counter['failed_preprocess'] += 1
            continue

        # --- 2. 预处理 (适配新版 utils) ---
        pre_calc_data = utils.prepare_molecule_for_cutting(mol)
        
        if not pre_calc_data: 
            stats_counter['failed_preprocess'] += 1
            continue

        main_mol = pre_calc_data['main_mol_for_matching']
        ion_fragments = pre_calc_data['ion_fragments'] # [新增] 提取离子
        
        final_covalent_fragments = []
        dropped_count = 0

        # --- 3. 执行切割 (增加保护逻辑) ---
        if main_mol is not None and main_mol.GetNumAtoms() > 0:
            final_covalent_fragments = utils.execute_fragmentation_pipeline(main_mol, pre_calc_data['bond_info'])
        else:
            # 如果主分子重构失败
            if main_mol is None and not ion_fragments:
                stats_counter['failed_preprocess'] += 1
                continue
            if main_mol is None:
                dropped_count += 1 # 记一次主分子丢失
        
        # --- 4. 计算描述符与存储 (增加 fallback) ---
        results_storage = []
        
        # [修订] 传入 fallback: main_mol if main_mol else mol，防止 None 传参
        d_count = utils.calculate_descriptors_and_store(
            storage=results_storage, mol_data={'IDs': compound_id, 'Smiles': smiles_string},
            original_mol_with_conf=mol, 
            main_mol_for_matching=main_mol if main_mol else mol, 
            final_covalent_fragments=final_covalent_fragments, 
            ion_fragments=ion_fragments,
            original_eccentricities=pre_calc_data['original_eccentricities'],
            original_logp_contribs=pre_calc_data['original_logp_contribs'],
            original_tpsa_contribs=pre_calc_data['original_tpsa_contribs'],
            original_charges=pre_calc_data['original_charges']
        )
        dropped_count += d_count
        
        # [新增] 质量控制: 检查是否丢失了过多片段
        if dropped_count > 5:
            stats_counter['failed_quality_control'] += 1
            continue
            
        if not results_storage:
             stats_counter['failed_quality_control'] += 1
             continue

        processed_result = results_storage[0]
        
        if not processed_result['results'] and not processed_result['ions']:
            stats_counter['failed_quality_control'] += 1
            continue

        # --- 5. 成功: 更新词汇表并保存结果 ---
        for item in processed_result['results']:
            clean_smi = Chem.MolToSmiles(item['clean_mol'])
            vocabulary.add(clean_smi)
            item['display_smiles'] = clean_smi
        for item in processed_result['ions']:
            vocabulary.add(item['smiles'])
            item['display_smiles'] = item['smiles']

        all_fragments_this_run.extend(processed_result['results'])
        all_fragments_this_run.extend(processed_result['ions'])
        
        all_molecules_processed_data.append({
            'mol_data': mol_data,
            'processed_result': processed_result
        })
        stats_counter['success'] += 1

    except Exception as e:
        print(f"\n! 异常: 处理分子 '{compound_id}' 时报错: {e}")
        stats_counter['failed_preprocess'] += 1

print("\n" + "="*50)
print(f"处理流程统计:")
print(f"  总输入分子: {stats_counter['total']}")
print(f"  成功保留  : {stats_counter['success']}")
print(f"  加载失败  : {stats_counter['failed_load']}")
print(f"  预处理失败: {stats_counter['failed_preprocess']}")
print(f"  质控丢弃  : {stats_counter['failed_quality_control']}")
print("="*50)

In [ ]:
# =============================================================================
# [新增] 辅助函数: ChemBERTa Embedding 生成
# =============================================================================

def get_chemberta_embedding(smiles):
    """
    输入 SMILES，返回 ChemBERTa 编码并降维后的 List。
    """
    if not smiles or smiles == "":
        return torch.zeros(TARGET_EMBEDDING_DIM).tolist()
        
    try:
        inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        
        if PROJECTION_MATRIX is not None:
            final_embedding = torch.matmul(cls_embedding, PROJECTION_MATRIX)
        else:
            final_embedding = cls_embedding
            
        return final_embedding.squeeze().tolist()
    except Exception as e:
        return torch.zeros(TARGET_EMBEDDING_DIM).tolist()

In [ ]:
# =============================================================================
# 模块 4: 加载全局归一化参数并更新词汇表 (ChemBERTa + JSON优化版)
# =============================================================================

print("归一化模式: 只读。正在加载已有的全局统计数据...")

if not norm_stats:
    raise RuntimeError(f"错误: 未能从 '{NORM_STATS_FILE}' 加载任何归一化统计数据。")

# --- 从加载的Welford统计量计算出全局的 mean 和 std ---
final_norm_params = {}
for prop, stats in norm_stats.items():
    count, mean, M2 = stats.get('count', 0), stats.get('mean', 0.0), stats.get('M2', 0.0)
    std = np.sqrt(M2 / count) if count >= 2 else 0.0
    final_norm_params[prop] = {'mean': mean, 'std': std}

print("成功从全局统计数据计算出 Mean / Std 参数。")


# 1. 识别新的SMILES并为它们生成嵌入
existing_smiles = set(vocab_data.keys())
new_smiles_to_add = vocabulary - existing_smiles

if new_smiles_to_add:
    sorted_new_smiles = sorted(list(new_smiles_to_add))
    print(f"\n正在为 {len(sorted_new_smiles)} 个新片段生成 ChemBERTa Embeddings...")
    
    # 使用 tqdm 显示生成进度
    for smi in tqdm(sorted_new_smiles, desc="Generating Embeddings"):
        # [关键修订] 调用 ChemBERTa 模型生成并降维
        vocab_data[smi] = get_chemberta_embedding(smi)
        
    print(f"新增片段处理完成。")

# 2. 确保 "<UNK>" 存在
if "<UNK>" not in vocab_data:
    vocab_data["<UNK>"] = torch.zeros(TARGET_EMBEDDING_DIM).tolist()

# 3. [修订] 保存词汇表 (使用自定义格式: 单行列表)
# 为了保存，先将 <UNK> 提取出来，单独处理
unk_embedding = vocab_data.pop("<UNK>")
sorted_items = sorted(vocab_data.items())

try:
    print(f"正在保存词汇表至 '{VOCAB_FILE}' ...")
    with open(VOCAB_FILE, 'w', encoding='utf-8') as f:
        f.write("{\n")
        # 手动写入第一行 <UNK>
        f.write(f'  "<UNK>": {json.dumps(unk_embedding)}')
        
        if sorted_items:
            f.write(",\n")
            for i, (smi, vec) in enumerate(sorted_items):
                # key 使用 ensure_ascii=False 保证可读性
                # value 使用 json.dumps 将列表转为单行字符串
                line = f'  {json.dumps(smi, ensure_ascii=False)}: {json.dumps(vec)}'
                
                # 处理逗号和换行
                if i < len(sorted_items) - 1:
                    f.write(f"{line},\n")
                else:
                    f.write(f"{line}\n")
        else:
            f.write("\n")
        f.write("}")
    
    print(f"共享词汇表已成功更新并保存。最终大小: {len(sorted_items) + 1}")

except Exception as e:
    print(f"保存词汇表时发生错误: {e}")
finally:
    # [重要] 无论保存是否成功，都必须将 <UNK> 放回内存中的字典
    # 否则后续模块5在查表时会报错
    vocab_data["<UNK>"] = unk_embedding

# --- 打印统计信息供参考 ---
print(f"注意: 本次运行不会修改归一化统计文件 '{NORM_STATS_FILE}'。")
print("\n--- 基于加载的全局统计量计算出的归一化参数 (Mean / Std) ---")
if not final_norm_params:
    print("没有可计算的数值属性。")
else:
    feature_columns_2d = [
        'avg_ecc', 'ecc_range', 'avg_logp', 'avg_tpsa', 'avg_charge', 
        'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable'
    ]
    for prop in feature_columns_2d:
        if prop in final_norm_params:
            params = final_norm_params[prop]
            print(f"{prop:<20}: Mean = {params['mean']:.4f}, Std = {params['std']:.4f}")

In [ ]:
# =============================================================================
# 模块 5: 组装并保存PyG Data对象 (修订版: 三类边逻辑)
# =============================================================================

print("\n" + "="*80)
print("开始组装并保存PyG Data对象...")

feature_columns = sorted([
    'avg_ecc', 'ecc_range', 'avg_logp', 'avg_tpsa', 'avg_charge', 
    'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable'
])

unk_embedding = vocab_data.get("<UNK>", torch.zeros(EMBEDDING_DIM).tolist())
saved_files_count = 0

for data_pack in all_molecules_processed_data:
    
    mol_data = data_pack['mol_data']
    processed_result = data_pack['processed_result']
    compound_id = mol_data['分子ID']
    
    try:
        # --- 1. 准备节点特征 (data.x) ---
        all_fragments = processed_result['results'] + processed_result['ions']
        if not all_fragments: continue

        df_frags = pd.DataFrame(all_fragments)
        
        # 提取数值特征
        df_frags_numeric = df_frags[feature_columns].copy()
        
        # 归一化策略分类 (与 SE3TD PDBbind 脚本保持一致)
          COUNT_FEATURES = {'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable'}

          # 执行归一化
          for prop, params in final_norm_params.items():
              if prop in df_frags_numeric.columns:
                  mean, std = params['mean'], params['std']

                  if prop in COUNT_FEATURES:
                      # 策略 B: 计数特征 — std 最小保底 1.0，防止零膨胀爆表
                      safe_std = max(std, 1.0)
                      df_frags_numeric[prop] = ((df_frags_numeric[prop] - mean) / safe_std).clip(-6.0, 6.0)
                  else:
                      # 策略 C: 连续型理化特征 — 标准 Z-score + 物理钳制
                      safe_std = std if std > 1e-5 else 1.0
                      df_frags_numeric[prop] = ((df_frags_numeric[prop] - mean) / safe_std).clip(-6.0, 6.0)
        
        x = torch.tensor(df_frags_numeric.values, dtype=torch.float)

        # --- 2. 准备边的连接和属性 (edge_index, edge_attr) ---
        # [修订逻辑开始]
        fragment_counter = 0
        covalent_fragment_ids = []
        ion_fragment_ids = []
        atom_to_fragment_map = {}

        # 2.1 建立 ID 映射
        for item in processed_result['results']:
            item['fragment_id'] = fragment_counter
            covalent_fragment_ids.append(fragment_counter)
            for atom in item['mol'].GetAtoms():
                if atom.HasProp("_original_index"):
                    atom_to_fragment_map[atom.GetIntProp("_original_index")] = fragment_counter
            fragment_counter += 1
        
        for item in processed_result['ions']:
            item['fragment_id'] = fragment_counter
            ion_fragment_ids.append(fragment_counter)
            fragment_counter += 1
        
        # 2.2 准备原始分子与键信息
        original_mol = Chem.MolFromSmiles(mol_data['分子smiles式'])
        # 防止 None 传入
        if original_mol is None:
             print(f"  警告: 无法为 {compound_id} 重建原始分子，跳过。")
             continue

        pre_calc_data_for_bonds = utils.prepare_molecule_for_cutting(original_mol)
        
        main_mol_bonds = set()
        if pre_calc_data_for_bonds and pre_calc_data_for_bonds['main_mol_for_matching'] is not None:
            main_mol_bonds = {tuple(sorted((b.GetBeginAtomIdx(), b.GetEndAtomIdx()))) 
                              for b in pre_calc_data_for_bonds['main_mol_for_matching'].GetBonds()}
        
        final_connections = set()
        for item in processed_result['results']:
            for bond in item['mol'].GetBonds():
                if bond.GetBeginAtom().GetAtomicNum() != 0 and bond.GetEndAtom().GetAtomicNum() != 0:
                    final_connections.add(tuple(sorted((bond.GetBeginAtom().GetIntProp("_original_index"), 
                                                        bond.GetEndAtom().GetIntProp("_original_index")))))
        
        # 计算被切断的键 (即共价连接边)
        severed_bonds = main_mol_bonds - final_connections
        
        edge_list = []
        edge_attr_list = []
        existing_covalent_pairs = set() # 用于记录已存在的共价连接，避免全局边重复覆盖

        # --- A. 创建共价边 (Covalent Edge) ---
        # 逻辑: 原分子中有键连接，但现在分属不同片段
        # 属性: [1.0, 0.0, 0.0]
        for bond_tuple in severed_bonds:
            id1 = atom_to_fragment_map.get(bond_tuple[0])
            id2 = atom_to_fragment_map.get(bond_tuple[1])
            if id1 is not None and id2 is not None and id1 != id2:
                pair = tuple(sorted((id1, id2)))
                # 避免多重键导致重复添加边
                if pair not in existing_covalent_pairs:
                    edge_list.append(list(pair))
                    edge_attr_list.append([1.0, 0.0, 0.0]) # Type 1
                    existing_covalent_pairs.add(pair)

        # --- B. [新增] 创建全局边 (Global Edge) ---
        # 逻辑: 两个共价片段如果不相邻(没有共价边)，则建立全局边
        # 属性: [0.0, 1.0, 0.0]
        if len(covalent_fragment_ids) > 1:
            for id1, id2 in itertools.combinations(covalent_fragment_ids, 2):
                pair = tuple(sorted((id1, id2)))
                if pair not in existing_covalent_pairs:
                    edge_list.append(list(pair))
                    edge_attr_list.append([0.0, 1.0, 0.0]) # Type 2

        # --- C. 创建离子边 (Ionic Edge) ---
        # 逻辑: 离子片段与所有共价片段全连接
        # 属性: [0.0, 0.0, 1.0]
        if ion_fragment_ids:
            for ion_id in ion_fragment_ids:
                for covalent_id in covalent_fragment_ids:
                    edge_list.append(sorted([ion_id, covalent_id]))
                    edge_attr_list.append([0.0, 0.0, 1.0]) # Type 3

        # 2.3 构建 Tensor 并处理无向图
        if edge_list:
            edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)
            
            # 双向化 (Undirected Graph)
            edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
            edge_attr = torch.cat([edge_attr, edge_attr], dim=0)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            # [重要] 维度修正为 3
            edge_attr = torch.empty((0, 3), dtype=torch.float)

        # --- 3. 准备标签 (y) ---
        y_values = [mol_data.get(col, np.nan) for col in activity_columns]
        y = torch.tensor([y_values], dtype=torch.float)

        # --- 4. 准备片段 Embedding ---
        frag_embeds_list = []
        for item in all_fragments:
            smi = item.get('display_smiles', '<UNK>')
            embedding = vocab_data.get(smi, unk_embedding)
            frag_embeds_list.append(embedding)
        frag_embeds = torch.tensor(frag_embeds_list, dtype=torch.float)

        # --- 5. 组装Data对象 ---
        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=y,
            mol_id=compound_id,
            smiles=mol_data['分子smiles式'],
            frag_embeds=frag_embeds 
        )
        
        # --- 6. 保存到文件 ---
        save_path = os.path.join(PYG_DATA_DIR, f"{compound_id}.pt")
        torch.save(data, save_path)
        saved_files_count += 1

    except Exception as e:
        print(f"  ! 在为分子 {compound_id} 组装或保存Data对象时出错: {e}")
        # import traceback
        # traceback.print_exc()

print(f"\n处理完成。共成功保存了 {saved_files_count} 个PyG Data对象。")

In [ ]:
# =============================================================================
# 模块 6: (验证用) 加载并检查单个PyG Data对象 (PyTorch 2.6+ 兼容版)
# =============================================================================

if all_molecules_processed_data:
    first_molecule_id = all_molecules_processed_data[0]['mol_data']['分子ID']
    file_to_check = os.path.join(PYG_DATA_DIR, f"{first_molecule_id}.pt")

    print("\n" + "="*80)
    print(f"正在加载并检查文件: '{file_to_check}'")
    print("="*80)

    try:
        loaded_data = torch.load(file_to_check, weights_only=False)

        # ... (打印 data.x, edge_index, edge_attr, y 的部分保持不变) ...
        print("\n--- 节点特征: data.x ---")
        print(f"Shape: {loaded_data.x.shape}")
        print(loaded_data.x)
        print("\n--- 边的连接: data.edge_index ---")
        print(f"Shape: {loaded_data.edge_index.shape}")
        print(loaded_data.edge_index)
        print("\n--- 边的属性: data.edge_attr ---")
        print(f"Shape: {loaded_data.edge_attr.shape}")
        print(loaded_data.edge_attr)
        print("\n--- 图级别标签: data.y ---")
        print(f"Shape: {loaded_data.y.shape}")
        print(loaded_data.y)
        
        print("\n--- 其他元数据 ---")
        print(f"分子ID (mol_id): {loaded_data.mol_id}")
        print(f"原始SMILES (smiles): {loaded_data.smiles}")
        
        # --- *** 修改: 检查并打印片段嵌入 *** ---
        print("\n--- 片段嵌入 (frag_embeds) ---")
        if hasattr(loaded_data, 'frag_embeds'):
            print(f"Shape: {loaded_data.frag_embeds.shape}")
            print("内容 (每个节点的嵌入向量):")
            print(loaded_data.frag_embeds)
        else:
            print("未在Data对象中找到 'frag_embeds' 属性。")
        # ***************************************

    except FileNotFoundError:
        print(f"\n错误: 未找到验证文件 '{file_to_check}'。")
    except Exception as e:
        print(f"\n加载或检查文件时发生错误: {e}")

else:
    print("\n" + "="*80)
    print("本次运行没有成功处理任何分子，无法进行验证。")